## Great Lakes Dashboard

In [21]:
import pandas as pd
import plotly.express as px

In [27]:
# load in csv

joyce_data = pd.read_csv('../data/R Results_Updated(Indirect Water Use Table).csv')
coeff_and_gen_data = pd.read_csv('../data/Coefficent and Generation Table.csv') 

In [4]:
#print columns
#take away spaces
joyce_data.columns = joyce_data.columns.str.strip()
joyce_data["State"] = joyce_data["State"].str.strip()
print(joyce_data.columns)

Index(['Scenario', 'State', 'Generation 2025 (MWh)', 'Generation 2030 (MWh)',
       'Change in Generation (MWh)', 'Indirect Water Use 2025 (gallons)',
       'Indirect Water Use 2030 (gallons)',
       'Change in Indirect Water Use (gallons)',
       'Indirect Water Use 2025 (liters)', 'Indirect Water Use 2030 (liters)',
       'Change in Indirect Water Use (liters)',
       'Direction of Water Use Change'],
      dtype='object')


In [79]:
coeff_and_gen_data.columns = coeff_and_gen_data.columns.str.strip()
coeff_and_gen_data["State"] = coeff_and_gen_data["State"].str.strip()
#strip white space from measure column in coeff_and_gen_data
coeff_and_gen_data["measure"] = coeff_and_gen_data["measure"].str.strip()
#strip total column
coeff_and_gen_data["total"] = coeff_and_gen_data["total"].str.strip()

### Regional Overview
**Total Data Center Electricity Demand**

In [5]:
# turn the number columns from text like ' 31,290,777.33 ' into real numbers
num_cols = joyce_data.columns.drop(["State", "Direction of Water Use Change"])
for col in num_cols:
    joyce_data[col] = pd.to_numeric(
        joyce_data[col].astype(str)
            .str.replace(",", "")      # 31,290,777 -> 31290777
            .str.replace("(", "-")     # (2578460304.84) -> -2578460304.84)
            .str.replace(")", "")      # -2578460304.84) -> -2578460304.84
            .str.strip()               # remove the extra spaces
    )

joyce_data.dtypes

Scenario                                    int64
State                                      object
Generation 2025 (MWh)                     float64
Generation 2030 (MWh)                     float64
Change in Generation (MWh)                float64
Indirect Water Use 2025 (gallons)         float64
Indirect Water Use 2030 (gallons)         float64
Change in Indirect Water Use (gallons)    float64
Indirect Water Use 2025 (liters)          float64
Indirect Water Use 2030 (liters)          float64
Change in Indirect Water Use (liters)     float64
Direction of Water Use Change              object
dtype: object

In [6]:
#list unique states
joyce_data["State"].unique()

array(['Illinois', 'Indiana', 'Michigan', 'Minnesota', 'New York', 'Ohio',
       'Pennsylvania', 'Wisconsin', 'Total'], dtype=object)

In [7]:
# total data center electricity demand in the Great Lakes Region in 2025 and 2030
totals = joyce_data[joyce_data["State"] == "Total"]
row = totals.iloc[0] #same in every scenario so any row works

demand = pd.DataFrame({
    "Year": ["2025", "2030"],
    "Demand (TWh)": [row["Generation 2025 (MWh)"]/1e6, row["Generation 2030 (MWh)"]/1e6],
})

fig_demand = px.bar(demand, x="Year", y="Demand (TWh)", title="Total Data Center Electricity Demand in the Great Lakes Region", text_auto=".1f")

fig_demand.show()
fig_demand.write_html("electricity_demand.html", include_plotlyjs="cdn")  # saves it as a file

**Total Indirect Water Use Each Year**

In [8]:
totals = joyce_data[joyce_data["State"] == "Total"]
print(totals.head)

<bound method NDFrame.head of     Scenario  State  Generation 2025 (MWh)  Generation 2030 (MWh)  \
8          1  Total            31290777.33           1.386237e+08   
17         2  Total            31290777.33           1.386237e+08   
26         3  Total            31290777.33           1.386237e+08   
35         4  Total            31290777.33           1.386237e+08   
44         5  Total            31290777.33           1.386237e+08   
53         6  Total            31290777.33           1.386237e+08   
62         7  Total            31290777.33           1.386237e+08   
71         8  Total            31290777.33           1.386237e+08   

    Change in Generation (MWh)  Indirect Water Use 2025 (gallons)  \
8                 1.073330e+08                       2.860507e+10   
17                1.073330e+08                       2.860507e+10   
26                1.073330e+08                       2.866679e+10   
35                1.073330e+08                       2.857718e+10   
44 

In [9]:

# total indirect water use in each year as a range from across scenarios

import plotly.graph_objects as go

scenario_names = {1: "High Demand", 2: "Same energy mix", 3: "Mid case", 4: "High renewable cost", 5: "Low renewable cost", 6: "Low gas price", 7: "Low RE cost + high gas", 8: "High RE cost + low gas"}

fig_water = go.Figure()
scenarios = totals["Scenario"].tolist()

#one set of bars per scenario, and show only the first one at start
for i, s in enumerate(scenarios):
    row = totals[totals["Scenario"] == s].iloc[0]
    fig_water.add_trace(go.Bar(
        x=["2025", "2030"],
        y=[row["Indirect Water Use 2025 (gallons)"] / 1e9,
           row["Indirect Water Use 2030 (gallons)"] / 1e9],
        name=scenario_names[s],
        visible=(i == 0)  # Show only the first scenario initially

    ))
buttons = []
for i, s in enumerate(scenarios):
    visible = [j == i for j in range(len(scenarios))]
    buttons.append(dict(label=scenario_names[s], method="update", args=[{"visible": visible}]))
    
fig_water.update_layout(
    title="Great Lakes indirect water use",
    yaxis_title="Billion gallons",
    updatemenus=[dict(buttons=buttons, x=0, xanchor="left", y=1.15, yanchor="top")],
)

fig_water.show()
fig_water.write_html("regional_water.html", include_plotlyjs="cdn")

**Overall Change from 2025 to 2030 for Electricity and Indirect Water Use**

In [10]:
# demand change (the same for every scenario)
d25, d30 = demand["Demand (TWh)"]
print(f"Demand: {d25:.1f} -> {d30:.1f} TWh, +{d30 - d25:.1f} TWh (+{(d30 - d25) / d25 * 100:.0f}%)")

# water change (different for each scenario, so report the range)
w25 = totals["Indirect Water Use 2025 (gallons)"] / 1e9
w30 = totals["Indirect Water Use 2030 (gallons)"] / 1e9
change = w30 - w25
pct = change / w25 * 100
print(f"Water use change: +{change.min():.1f} to +{change.max():.1f} billion gal "
      f"(+{pct.min():.0f}% to +{pct.max():.0f}%)")

Demand: 31.3 -> 138.6 TWh, +107.3 TWh (+343%)
Water use change: +55.4 to +91.9 billion gal (+193% to +321%)


### Across States
**2030 energy generation demand between states**

In [101]:
# %%

# 2030 energy generation demand between states 
gen_2030 = coeff_and_gen_data[coeff_and_gen_data["measure"] == "Generation 2030 (MWh)"].copy()

gen_2030["total"] = pd.to_numeric(
    gen_2030["total"].astype(str).str.replace(",", ""), errors="coerce"
)

#one value per state since none change across scenario
gen_by_state = gen_2030.groupby("State", as_index=False)["total"].first()

#states
states = gen_by_state["State"]
fig_gen_2030 = go.Figure()

fig_gen_2030.add_trace(go.Bar(
    x=states,
    y=gen_by_state["total"] / 1e6  # Convert MWh to TWh
))

fig_gen_2030.update_layout(
    title=f"2030 Energy Generation Demand by State",
    yaxis_title="Demand (TWh)",
)

fig_gen_2030.show()
fig_gen_2030.write_html("2030_energy_generation.html", include_plotlyjs="cdn")

**How technology mix changes for each state scross each scenario**

biomass = biopower/landfill gas

canada = Canadian electricity imports

coal = coal generation

csp = concentrating solar power

distpv = behind-the-meter distributed solar PV

gas-cc = natural-gas combined cycle

gas-ct = natural-gas combustion turbine

geothermal = geothermal

hydro = hydropower

nuclear = conventional nuclear

o-g-s = oil-gas-steam

phs = pumped-hydropower storage

upv = utility-scale/distributed-utility-scale solar PV

wind-ons = onshore wind

wind-ofs = offshore wind

In [110]:
#turn each column name in tech_mix_factors into a float
tech_mix_factors = ["biomass", "canada", "coal", "csp", "distpv", "gas-cc", "gas-ct", "geothermal", "hydro", "nuclear", "o-g-s", "phs", "upv", "wind-ons", "wind-ofs"]

#for
for factor in tech_mix_factors:
    gen_2030[factor] = pd.to_numeric(
        gen_2030[factor].astype(str).str.replace(",", ""), errors="coerce"
    )
gen_2030.dtypes

Scenario_ID      int64
Scenario        object
State           object
measure         object
biomass        float64
canada         float64
coal           float64
csp            float64
distpv         float64
gas-cc         float64
gas-ct         float64
geothermal     float64
hydro          float64
nuclear        float64
o-g-s          float64
phs            float64
upv            float64
wind-ons       float64
wind-ofs       float64
total          float64
Unnamed: 20    float64
dtype: object

In [113]:
fig_tech_mix = go.Figure()
n_tech = len(tech_mix_factors)

#get the technology mix from 

for i, s in enumerate(scenarios):
    sub = gen_2030[gen_2030["Scenario_ID"] == s].sort_values("State")
    for tech in tech_mix_factors:
        fig_tech_mix.add_trace(go.Bar(
            x=sub["State"],
            y=sub[tech] / 1e6,  # Convert MWh to TWh
            name=tech,
            showlegend=(i == 0),  # Show legend only for the first scenario
            visible=(i == 0),  # Show only the first scenario initially
            hovertemplate=f"{tech}: " + "%{y:.2f} TWh<extra></extra>",
    ))
buttons = []
for i, s in enumerate(scenarios):
    visible = [False] * (len(scenarios) * n_tech)
    visible[i * n_tech:(i + 1) * n_tech] = [True] * n_tech   # turn on this scenario's 15 traces
    buttons.append(dict(
        label=scenario_names[s], method="update", args=[{"visible": visible}, 
              {"title": f"2030 Generation Mix by State: {scenario_names[s]}"}],
              ))

fig_tech_mix.update_layout(
    barmode="stack",
    title=f"2030 Generation Mix by State: {scenario_names[scenarios[0]]}",
    xaxis_title="State",
    yaxis_title="Demand (TWh)",
    updatemenus=[dict(buttons=buttons, x=0, xanchor="left", y=1.15, yanchor="top")],
)

fig_tech_mix.show()
fig_tech_mix.write_html("2030_energy_generation_tech_mix.html",include_plotlyjs="cdn")
    

### Convert to HTML

In [ ]:
# turn each chart into piece of HTML
demand_html = fig_demand.to_html(full_html=False, include_plotlyjs="cdn", default_width="100%", default_height="420px", config={"responsive": True})
water_html = fig_water.to_html(full_html=False, include_plotlyjs=False, default_width="100%", default_height="420px", config={"responsive": True})
gen_html = fig_gen_2030.to_html(full_html=False, include_plotlyjs=False, default_width="100%", default_height="420px", config={"responsive": True})
mix_html = fig_tech_mix.to_html(full_html=False, include_plotlyjs=False, default_width="100%", default_height="520px", config={"responsive": True})

# put them on one page
page = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>Great Lakes Data Center Water & Energy Impacts</title>
  <style>
    body {{ font-family: system-ui, sans-serif; margin: 0; background: #f4f5f3; }}
    header {{ background: #16304f; color: white; padding: 16px 24px; }}
    header h1 {{ margin: 0; font-size: 22px; }}
    header p {{ margin: 4px 0 0; color: #c4d3e6; }}
    main {{ padding: 24px; max-width: 1200px; margin: auto; }}
    .row {{ display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }}
    @media (max-width: 800px) {{ .row {{ grid-template-columns: 1fr; }} }}
    .card {{ background: white; border-radius: 10px; padding: 16px; min-width: 0; }}
    .big {{ font-size: 28px; font-weight: bold; }}
  </style>
</head>
<body>
  <header>
    <h1>Great Lakes Data Center Water & Energy Impacts</h1>
    <p>How electricity demand from data centers translates into water use</p>
  </header>
  <main>
    <h2>Regional overview</h2>
    <div class="row">
      <div class="card">{demand_html}</div>
      <div class="card">{water_html}</div>
    </div>
    <div class="row" style="margin-top:16px">
      <div class="card">
        <div>Change in electricity demand, 2025–2030</div>
        <div class="big">+{d30 - d25:.1f} TWh</div>
        <div>+{(d30 - d25) / d25 * 100:.0f}% (same in every scenario)</div>
      </div>
      <div class="card">
        <div>Change in indirect water use, 2025–2030</div>
        <div class="big">+{change.min():.1f} to +{change.max():.1f} billion gal</div>
        <div>+{pct.min():.0f}% to +{pct.max():.0f}%, depending on scenario</div>
      </div>
    </div>
    <h2 style="margin-top:32px">Across states</h2>
    <div class="row">
      <div class="card">{gen_html}</div>
      <div class="card">
        <!-- placeholder: 2030 indirect water use by state goes here -->
      </div>
    </div>
    <div class="card" style="margin-top:16px">{mix_html}</div>
  </main>
</body>
</html>"""

import os
os.makedirs("../docs", exist_ok=True)   # make docs folder next to scripts/

with open("../docs/index.html", "w", encoding="utf-8") as f:
    f.write(page)
print("Saved docs/index.html")

Saved docs/index.html
